## HMM

In [2]:
import numpy as np
class SimpleHMM:
    """
    一个简单的离散型 HMM 类。
    模型参数：
        A: 状态转移概率矩阵，shape = (N, N)
        B: 观测概率矩阵，shape = (N, M)
        pi: 初始状态概率，shape = (N,)
    其中：
        N 表示隐藏状态个数
        M 表示观测符号个数
    """
    def __init__(self, n_states, n_obs, A=None, B=None, pi=None, random_state=0):
        """
        初始化 HMM 模型。
        参数：
            n_states: 隐藏状态数量
            n_obs: 观测符号数量
            A: 状态转移矩阵
            B: 观测概率矩阵
            pi: 初始状态概率
            random_state: 随机种子
        """
        self.n_states = n_states
        self.n_obs = n_obs
        rng = np.random.default_rng(random_state)
        if A is None:
            A = rng.random((n_states, n_states))
            A = A / A.sum(axis=1, keepdims=True)
        if B is None:
            B = rng.random((n_states, n_obs))
            B = B / B.sum(axis=1, keepdims=True)
        if pi is None:
            pi = rng.random(n_states)
            pi = pi / pi.sum()
        self.A = np.array(A, dtype=float)
        self.B = np.array(B, dtype=float)
        self.pi = np.array(pi, dtype=float)

    def forward(self, O):
        """
        前向算法。
        输入：
            O: 观测序列，例如 [0, 1, 0, 1]
        返回：
            alpha: 前向概率矩阵
            prob: 观测序列概率 P(O | lambda)
        alpha[t, i] 表示：
            在 t 时刻处于隐藏状态 i，并且已经观测到 O[0],...,O[t] 的概率。
        """
        O = np.array(O, dtype=int)
        T = len(O)
        N = self.n_states
        alpha = np.zeros((T, N))
        # 第 1 步：初始化
        alpha[0] = self.pi * self.B[:, O[0]]
        # 第 2 步：递推
        for t in range(1, T):
            for j in range(N):
                alpha[t, j] = np.sum(alpha[t - 1] * self.A[:, j]) * self.B[j, O[t]]
        # 第 3 步：终止
        prob = np.sum(alpha[T - 1])
        return alpha, prob

    def backward(self, O):
        """
        后向算法。
        输入：
            O: 观测序列，例如 [0, 1, 0, 1]
        返回：
            beta: 后向概率矩阵
            prob: 观测序列概率 P(O | lambda)
        beta[t, i] 表示：
            已知 t 时刻处于隐藏状态 i，从 t+1 到 T 产生后续观测的概率。
        """
        O = np.array(O, dtype=int)
        T = len(O)
        N = self.n_states
        beta = np.zeros((T, N))
        # 第 1 步：初始化
        beta[T - 1] = 1.0
        # 第 2 步：递推
        for t in range(T - 2, -1, -1):
            for i in range(N):
                beta[t, i] = np.sum(self.A[i, :] * self.B[:, O[t + 1]] * beta[t + 1])
        # 第 3 步：终止
        prob = np.sum(self.pi * self.B[:, O[0]] * beta[0])
        return beta, prob

    def score(self, O):
        """
        计算观测序列概率 P(O | lambda)。
        """
        _, prob = self.forward(O)
        return prob

    def viterbi(self, O):
        """
        维特比算法。
        输入：
            O: 观测序列，例如 [0, 1, 0, 1]
        返回：
            best_path: 最可能的隐藏状态序列
            best_prob: 这条最优路径的概率
        作用：
            求使 P(I, O | lambda) 最大的隐藏状态序列 I。
        """
        O = np.array(O, dtype=int)
        T = len(O)
        N = self.n_states

        delta = np.zeros((T, N))
        psi = np.zeros((T, N), dtype=int)

        # 第 1 步：初始化
        delta[0] = self.pi * self.B[:, O[0]]
        psi[0] = 0

        # 第 2 步：递推
        for t in range(1, T):
            for j in range(N):
                values = delta[t - 1] * self.A[:, j]
                psi[t, j] = np.argmax(values)
                delta[t, j] = np.max(values) * self.B[j, O[t]]

        # 第 3 步：终止
        best_prob = np.max(delta[T - 1])
        best_last_state = np.argmax(delta[T - 1])

        # 第 4 步：回溯最优路径
        best_path = np.zeros(T, dtype=int)
        best_path[T - 1] = best_last_state

        for t in range(T - 2, -1, -1):
            best_path[t] = psi[t + 1, best_path[t + 1]]

        return best_path, best_prob

    def fit(self, O, n_iter=20):
        """
        Baum-Welch 学习算法，也就是 HMM 的 EM 算法。
        输入：
            O: 观测序列，例如 [0, 1, 0, 1, 1, 0]
            n_iter: 迭代次数
        作用：
            在只有观测序列、没有隐藏状态序列的情况下，
            估计模型参数 A, B, pi。
        """
        O = np.array(O, dtype=int)
        T = len(O)
        N = self.n_states
        M = self.n_obs

        eps = 1e-12

        for iteration in range(n_iter):
            # E 步：计算 alpha 和 beta
            alpha, prob = self.forward(O)
            beta, _ = self.backward(O)

            # gamma[t, i] 表示：
            # 给定整个观测序列 O，在 t 时刻处于状态 i 的概率
            gamma = np.zeros((T, N))
            for t in range(T):
                denominator = np.sum(alpha[t] * beta[t]) + eps
                for i in range(N):
                    gamma[t, i] = alpha[t, i] * beta[t, i] / denominator

            # xi[t, i, j] 表示：
            # 给定整个观测序列 O，在 t 时刻处于状态 i，
            # 并且在 t+1 时刻转移到状态 j 的概率
            xi = np.zeros((T - 1, N, N))
            for t in range(T - 1):
                denominator = 0.0
                for i in range(N):
                    for j in range(N):
                        denominator += (
                            alpha[t, i]
                            * self.A[i, j]
                            * self.B[j, O[t + 1]]
                            * beta[t + 1, j]
                        )

                denominator += eps

                for i in range(N):
                    for j in range(N):
                        xi[t, i, j] = (
                            alpha[t, i]
                            * self.A[i, j]
                            * self.B[j, O[t + 1]]
                            * beta[t + 1, j]
                            / denominator
                        )

            # M 步：更新 pi
            self.pi = gamma[0]

            # M 步：更新 A
            for i in range(N):
                for j in range(N):
                    numerator = np.sum(xi[:, i, j])
                    denominator = np.sum(gamma[:-1, i]) + eps
                    self.A[i, j] = numerator / denominator

            # M 步：更新 B
            for j in range(N):
                for k in range(M):
                    numerator = 0.0
                    denominator = np.sum(gamma[:, j]) + eps

                    for t in range(T):
                        if O[t] == k:
                            numerator += gamma[t, j]

                    self.B[j, k] = numerator / denominator

            # 防止因为数值误差导致每一行不是概率分布
            self.A = self.A / self.A.sum(axis=1, keepdims=True)
            self.B = self.B / self.B.sum(axis=1, keepdims=True)
            self.pi = self.pi / self.pi.sum()

            print(f"第 {iteration + 1} 次迭代，P(O|lambda) = {prob:.8f}")

    def print_params(self):
        """
        打印当前模型参数。
        """
        print("A 状态转移矩阵：")
        print(self.A)

        print("\nB 观测概率矩阵：")
        print(self.B)

        print("\npi 初始状态概率：")
        print(self.pi)

## 检测

In [5]:
# 观测编码
# 红 = 0
# 白 = 1

A = [
    [0.5, 0.2, 0.3],
    [0.3, 0.5, 0.2],
    [0.2, 0.3, 0.5]
]

B = [
    [0.5, 0.5],
    [0.4, 0.6],
    [0.7, 0.3]
]

pi = [0.2, 0.4, 0.4]

model = SimpleHMM(
    n_states=3,
    n_obs=2,
    A=A,
    B=B,
    pi=pi
)

O = [0, 1, 0, 1]

alpha, prob_forward = model.forward(O)
beta, prob_backward = model.backward(O)

print("前向算法得到的概率：", prob_forward)
print("后向算法得到的概率：", prob_backward)

O = [0, 1, 0, 1]

best_path, best_prob = model.viterbi(O)

print("最可能的隐藏状态序列：", best_path)
print("最优路径概率：", best_prob)

O = [0, 1, 0, 0, 1, 0, 1, 1]

model = SimpleHMM(
    n_states=3,
    n_obs=2,
    random_state=42
)

print("学习前参数：")
model.print_params()

model.fit(O, n_iter=10)

print("\n学习后参数：")
model.print_params()

前向算法得到的概率： 0.06009079999999999
后向算法得到的概率： 0.06009079999999999
最可能的隐藏状态序列： [2 1 1 1]
最优路径概率： 0.0030239999999999993
学习前参数：
A 状态转移矩阵：
[[0.37363326 0.21187196 0.41449478]
 [0.3946247  0.05329282 0.55208249]
 [0.45432561 0.46920314 0.07647125]]

B 观测概率矩阵：
[[0.54845925 0.45154075]
 [0.59005935 0.40994065]
 [0.64980045 0.35019955]]

pi 初始状态概率：
[0.2687178  0.65581605 0.07546615]
第 1 次迭代，P(O|lambda) = 0.00326445
第 2 次迭代，P(O|lambda) = 0.00453855
第 3 次迭代，P(O|lambda) = 0.00572024
第 4 次迭代，P(O|lambda) = 0.00759006
第 5 次迭代，P(O|lambda) = 0.00969328
第 6 次迭代，P(O|lambda) = 0.01130796
第 7 次迭代，P(O|lambda) = 0.01233495
第 8 次迭代，P(O|lambda) = 0.01301243
第 9 次迭代，P(O|lambda) = 0.01353844
第 10 次迭代，P(O|lambda) = 0.01402673

学习后参数：
A 状态转移矩阵：
[[0.31362764 0.286619   0.39975336]
 [0.38855976 0.03634312 0.57509711]
 [0.4561923  0.47473558 0.06907212]]

B 观测概率矩阵：
[[0.43945629 0.56054371]
 [0.99800913 0.00199087]
 [0.03445074 0.96554926]]

pi 初始状态概率：
[5.13897471e-05 9.99948610e-01 2.57290541e-10]
